Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


In [ ]:
def get_baseline_for_all_events(waveform, baseline_front=(0.0,0.2)):
    baseline_start_f = int(1000 * baseline_front[0])
    baseline_end_f = int(1000 * baseline_front[1])

    # baseline is calculated with raw waveform
    # unit: same as raw waveform
    baseline_mean_V = np.mean(waveform[:,baseline_start_f:baseline_end_f], axis=1)
    baseline_std_V = np.std(waveform[:,baseline_start_f:baseline_end_f], axis=1)

    return baseline_mean_V, baseline_std_V
        

In [ ]:
def rolling_window(array: np.ndarray, window_size:int, axis:int) -> np.ndarray:
    """
    Rolls a 1D array into a 2D array with a sliding window view.
    Args:
        array (np.ndarray): The input 1D array to be rolled.
        window_size (int): The size of the rolling window.
        axis (int): The axis along which to roll the array.
    Returns:
        np.ndarray: A 2D array where each row corresponds to a window of the original array.
    """
    ndim = array.ndim

    if not isinstance(array, np.ndarray):
        raise ValueError("Input must be a numpy array.")
    if axis > ndim - 1 or axis < 0:
        raise ValueError("Axis must be within the range of the array dimensions.")
    if not isinstance(window_size, int) or window_size <= 0 or window_size > array.shape[axis]:
        raise ValueError("Window size must be a positive integer.")
    
    # n.dim rolling window
    # expand array according to the rolling window
    expanded_array = np.lib.stride_tricks.sliding_window_view(array, window_size, axis=axis)
    # take the mean along the new dimension for the result
    roll_averaged_array = expanded_array.mean(axis=ndim) 

    # roll_averaged_array = np.convolve(array, np.ones(window_size)/window_size, mode='valid')


    return roll_averaged_array

In [ ]:
def get_board_channel(SiPM_channel: int, board_0_channels: np.array, board_1_channels: np.array) -> int:
    if SiPM_channel in board_0_channels: 
        board_channel = np.where(board_0_channels == SiPM_channel)[0]
    elif SiPM_channel in board_1_channels:  
        board_channel = np.where(board_1_channels == SiPM_channel)[0]
    else:
        raise ValueError(f"SiPM channel {SiPM_channel} not found in both boards.")

    return board_channel[0]


In [ ]:
def set_peak_info_from_waveform_info(waveform_info: WaveformInfo):
    PeakInfoList = []
    peak_info = PeakInfo()
    peak_info.set_info_from_dict(waveform_info.__dict__)

    for peak_id in range(int(waveform_info.n_peaks)):
        peak_info.peak_id = peak_id
        peak_info.peak_start_time_s = waveform_info.peak_start_time_s_array[peak_id]
        peak_info.peak_end_time_s = waveform_info.peak_end_time_s_array[peak_id]
        peak_info.peak_rel_start_time_s = waveform_info.peak_rel_start_time_s_array[peak_id]
        peak_info.peak_height_V = waveform_info.peak_height_V_array[peak_id]
        peak_info.peak_width_ns = waveform_info.peak_width_ns_array[peak_id]
        peak_info.peak_area_Vns = waveform_info.peak_area_Vns_array[peak_id]
        peak_info.peak_area_PE = waveform_info.peak_area_PE_array[peak_id]

        tmp  = deepcopy(peak_info.__dict__)
        PeakInfoList.append(tmp)

    return PeakInfoList

In [ ]:
def get_peak_level_data(all_runs_d2d: d2d.data,
                        md_full_path: str,
                        peak_merge_window_sample: int):
    
    single_info = WaveformInfo()
    single_info_list = []
    
    mask = all_runs_d2d.md_full_path == md_full_path
    # print(f"Processing run {md_full_path}")
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = False)
    # print(single_run.__dict__)

    assert len(single_run.channel) == 24, f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels."
    
    t1,t2,t3, t4 = 0, 0, 0, 0

    for board_id in np.unique(single_run.board):
        
        mask = single_run.board == board_id
        single_board = single_run.apply_mask(mask, inplace=False, dry = True)

        board_info = WaveformInfo()
        board_info.set_info_from_dict(single_board.get_common_info_dict())
        board_info.board_0_channels = np.array(json.loads(board_info.board_0_channels))
        board_info.board_1_channels = np.array(json.loads(board_info.board_1_channels))

        # convert string to array
        if isinstance(board_info.board_0_channels[0], str):
            board_info.board_0_channels = board_info.board_0_channels.apply(json.loads).apply(np.array)
        if isinstance(board_info.board_1_channels[0], str):
            board_info.board_1_channels = board_info.board_1_channels.apply(json.loads).apply(np.array)

        if isinstance(board_info.board_0_channels[0], str):
            if "," in board_info.board_0_channels[0]:
                board_info.board_0_channels = board_info.board_0_channels.apply(json.loads).apply(np.array)
            else:
                board_info.board_0_channels = board_info.board_0_channels.apply(lambda x: x.replace("  ", ","))
                board_info.board_0_channels = board_info.board_0_channels.apply(lambda x: x.replace("[ ", "["))
                board_info.board_0_channels = board_info.board_0_channels.apply(lambda x: x.replace(" ", ","))
                board_info.board_0_channels = board_info.board_0_channels.apply(json.loads).apply(np.array)

                board_info.board_1_channels = board_info.board_1_channels.apply(lambda x: x.replace("  ", ","))
                board_info.board_1_channels = board_info.board_1_channels.apply(lambda x: x.replace("[ ", "["))
                board_info.board_1_channels = board_info.board_1_channels.apply(lambda x: x.replace(" ", ","))
                board_info.board_1_channels = board_info.board_1_channels.apply(json.loads).apply(np.array)
                

        # single_info.set_info_from_dict(board_info)
        event_processor = EventProcessor(board_info)
        
        if board_id == 0:
            channel_list = board_info.board_0_channels
        else:
            channel_list = board_info.board_1_channels


        for channel_id in range(len(channel_list)):
            mask = (single_run.board == board_id) & (single_run.channel == channel_list[channel_id])
            single_channel = single_run.apply_mask(mask, inplace=False, dry = True)
            
            assert len(single_channel.channel) == 1, f"Channel {channel_id} in board {board_id} has {len(single_channel.channel)} channels, expected 1 channel."
            
            single_info.set_info_from_dict(single_channel.get_dict())
            board_channel = get_board_channel(channel_id, board_info.board_0_channels, board_info.board_1_channels)

            waveform_processor = event_processor.get_waveform_processor(board_channel)
            waveform = waveform_processor.filtered_wfs

            baseline, baseline_std = get_baseline_for_all_events(waveform)

            # print(f"Processing board {single_info.board}, channel {single_info.channel} in run {single_info.md_full_path}")

            for event_id in range(waveform.shape[0]):
                single_waveform = waveform[event_id,:]
                single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

                single_info.event_start_time_s = waveform_processor.event_time_s[event_id]
                single_info.event_id = event_id
                # print(f"Processing event {single_info.event_id} in run {single_info.md_full_path}, board {single_info.board}, channel {single_info.channel}")

                t0 = time.perf_counter()
                single_info.set_peaks_for_single_processed_waveform(
                    single_waveform, 
                    single_baseline, 
                    single_baseline_std,
                    threshold_sig=3, 
                    extend_sum_window=50,
                    peak_merge_window_sample=peak_merge_window_sample
                    # event_id=event_id
                )
                t1 += time.perf_counter() - t0              

                t0 = time.perf_counter()
                PeakInfoList = set_peak_info_from_waveform_info(single_info)
                t2 += time.perf_counter() - t0              

                # single_info_list.append(single_info.__dict__.copy())
                # tmp = deepcopy(single_info.__dict__)
                # single_info_list.append(tmp)

                t0 = time.perf_counter()
                single_info_list += PeakInfoList
                t3 += time.perf_counter() - t0

    print(f'Time taken: {t1:.6f} seconds for peak finding, '
          f'{t2:.6f} seconds for peak info setting, '
          f'{t3:.6f} seconds for appending peak info list')
    
    return (single_info_list, waveform, baseline, baseline_std)

In [ ]:
def get_event_time(all_runs_d2d: d2d.data,
                        md_full_path: str):
    
    board_event_time_list = []
    
    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = False)

    assert len(single_run.channel) == 24, f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels."
    
    for board_id in np.unique(single_run.board):
        
        mask = single_run.board == board_id
        single_board = single_run.apply_mask(mask, inplace=False, dry = True)

        board_info = WaveformInfo()
        board_info.set_info_from_dict(single_board.get_common_info_dict())

        event_processor = EventProcessor(board_info)
        
        waveform_processor = event_processor.get_waveform_processor(board_channel = 0)

        board_event_time_list.append(waveform_processor.event_time_s)



    return board_event_time_list

In [ ]:
def get_waveform_from_single_info(single_info):
    
    # assert len(single_info.channel) == 1, f"Expected 1 channel."
    event_processor = EventProcessor(single_info)
            
    board_channel = get_board_channel(single_info.channel, single_info.board_0_channels, single_info.board_1_channels)

    waveform_processor = event_processor.get_waveform_processor(board_channel)
    waveform = waveform_processor.filtered_wfs
    baseline, baseline_std = get_baseline_for_all_events(waveform)

    return (waveform, baseline, baseline_std)

### Import and Process Data

In [ ]:
df_SPE_position = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spe_position_LXe.csv",
                 delimiter=",")

df_SPE_position

In [ ]:
df = pd.read_csv(
    # "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250706_LXe_gain_info_single_channel.csv",
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


# convert string to array
if isinstance(df['board_0_channels'][0], str):
    if "," in df['board_0_channels'][0]:
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)
    else:
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)

        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(json.loads).apply(np.array)

print(len(df))
idx_nan = np.where((df['spe_position'].isna()) & (df['voltage_preamp1_V']<-46))[0]
# replace NaN values with SPE position
voltage_array = df.loc[idx_nan, 'voltage_preamp1_V']
channel_array = df.loc[idx_nan, 'channel']
print(len(idx_nan))
for idx, voltage, channel in zip(idx_nan, voltage_array, channel_array):
    # find the SPE position for the given voltage and channel
    df.loc[idx, 'spe_position'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position'].values
    df.loc[idx, 'spe_position_err'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position_err'].values


In [ ]:
#### Basic Data Selection

all_runs_d2d = d2d.data(df)


mask_data_taking_mode = (all_runs_d2d.data_taking_mode == "all_channels")
mask_na = ~np.isnan(all_runs_d2d.spe_position)
# mask = ~np.isnan(all_runs_d2d.gain)
mask_voltage = (all_runs_d2d.voltage_preamp1_V < -46) & (all_runs_d2d.voltage_preamp1_V > -52)
mask_post_trigger = (all_runs_d2d.post_trigger == 70)
mask_run_tag = (all_runs_d2d.run_tag == "LXe/tritium")
# mask_run_tag = (all_runs_d2d.run_tag == "LXe/gain_calibration")
mask_comment = (~util.vec_regex_search('trash', all_runs_d2d.comment)) & (~util.vec_regex_search('test', all_runs_d2d.comment))

mask = mask_data_taking_mode & mask_na & mask_run_tag & mask_voltage & mask_post_trigger & mask_comment
# & mask_voltage

all_runs_d2d.apply_mask(mask, inplace=True, dry = False)
all_run_list = np.unique(all_runs_d2d.md_full_path)

#### Run Selection

check how many runs have all 24 channel data, print a list

In [ ]:
count_successful = 0
count_failed = 0

count_n_events = 0

for i, md_full_path in enumerate(all_run_list):

    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = True)

    if len(single_run.channel) < 24:
        # print(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels. Run tag: {single_run.run_tag[0]}. Comment: {single_run.comment[0]}. Voltage: {single_run.voltage_preamp1_V[0]}")
        count_failed += 1

        continue    
    elif len(single_run.channel) == 24:
        print(f"Run {i}: {md_full_path} has 24 channels. Run tag: {single_run.run_tag[0]}. Voltage: {single_run.voltage_preamp1_V[0]}. Comment: {single_run.comment[0]}. Event number: {single_run.n_processed_events[0]}")
        count_successful += 1
        count_n_events += single_run.n_processed_events[0]
        
    else: 
        raise ValueError(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")

print(f"Total runs: {len(all_run_list)}"
      f"Successful runs: {count_successful} "
      f"Failed runs: {count_failed} "
      f"Success rate: {count_successful/len(all_run_list)*100:.2f}%")

print(f"Total number of events: {count_n_events}")

#### Process Run

In [ ]:
### Choose a run from the list above
# initialize the result storage
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
output_fname = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse_3.csv"


# for run_id in np.arange(len(all_run_list)):
for run_id in [86,87,88,89,90]:
    single_info_list = []
    for md_full_path in all_run_list[run_id:run_id+1]:
    # for md_full_path in all_run_list:
        result, waveform, baseline, baseline_std = get_peak_level_data(
            all_runs_d2d=all_runs_d2d,
            md_full_path=md_full_path,
            peak_merge_window_sample=0
        )
        single_info_list += result

    df_result = pd.DataFrame.from_dict(single_info_list)

    df_result['first_peak_start_time'] = df_result.groupby('event_id')['peak_rel_start_time_s'].transform('min')
    df_result['time_diff'] = df_result['peak_rel_start_time_s'] - df_result['first_peak_start_time']

    # peak_rel_start_time = d2d_data.peak_rel_start_time_s
    time_diff = df_result['time_diff']

    hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
    bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    hist_log = np.log10(hist + 1)  # add 1 to avoid log(0)
    # ax.plot(bin_centers, hist_log, 'b-', label='Data')

    # fit the decay trend
    mask = (bin_centers >= 1) & (bin_centers <= 2.5)
    # linear fit
    p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


    # find tau
    tau = -1 / p[0]
    a = np.exp(p[1])
    # error of tau
    tau_err = np.sqrt(cov[0, 0]) / p[0]**2


    assert len(np.unique(df_result.md_full_path)) == 1


    df_output = pd.DataFrame({
        'file_name': df_result.md_full_path[0],
        'run_tag': df_result.run_tag[0],
        'comment': df_result.comment[0],
        'datetime': df_result.date_time[0],
        'voltage': np.unique(df_result.voltage_preamp1_V)[0],
        'tau': tau,
        'error': tau_err
    },
    index=[0]
    )


    with open(output_fname, 'a') as f:
        df_output.to_csv(f, mode='a', header=f.tell() == 0, index=False)

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_decay_time.pdf")


In [ ]:
afterpulse_df = pd.read_csv(
    '/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse_3.csv',
    parse_dates=["datetime"])
afterpulse_df['date'] = afterpulse_df['datetime'].dt.date


In [ ]:
len(afterpulse_df)

In [ ]:
afterpulse_df['tau'].mean()

In [ ]:
afterpulse_df['tau'].std()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))


for voltage in np.unique(afterpulse_df['voltage']):
    mask = afterpulse_df['voltage'] == voltage
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = (afterpulse_df['datetime'][mask] - afterpulse_df['datetime'][mask].iloc[0]).dt.total_seconds()


    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))
    print(f"Correlation between Tau and Time: {corr:.2f} ± {corr_err:.2f}")

    plt.errorbar(
        afterpulse_df['datetime'][mask],
        y_var, 
        yerr=y_var_err, 
        label=f'{voltage} V: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        )
    
plt.legend(
    title="Bias voltage: correlation",
    bbox_to_anchor=(0., 1),
    loc='upper left',
    frameon=True
)
plt.xlabel("Date")
plt.ylabel("$\\tau$ [$\mu$s]")

# fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_vs_time.pdf"
# plt.savefig(fname, dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

for date in np.unique(afterpulse_df['date']):
    mask = afterpulse_df['date'] == date
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = afterpulse_df['voltage'][mask]

    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))
    print(f"Correlation between Tau and Time: {corr:.2f} ± {corr_err:.2f}")

    plt.errorbar(
        x_var,
        y_var, 
        yerr=y_var_err, 
        label=f'{date}: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        )
    
plt.legend(
    title="Date: correlation",
    bbox_to_anchor=(0., 0.),
    loc='lower left',
    frameon=True
)
plt.xlabel("Bias voltage")
plt.ylabel("$\\tau$ [$\mu$s]")

plt.ylim(2., 5.2)


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15, 6), sharey=True)
plt.style.use('tableau-colorblind10')
colors = plt.rcParams["axes.prop_cycle"]()
fig.subplots_adjust(wspace=0)

for voltage in np.unique(afterpulse_df['voltage'][afterpulse_df['voltage']>-52]):
    mask = afterpulse_df['voltage'] == voltage
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = (afterpulse_df['datetime'][mask] - afterpulse_df['datetime'][mask].iloc[0]).dt.total_seconds()


    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))


    color = next(colors)["color"]


    ax[0].errorbar(
        afterpulse_df['datetime'][mask],
        y_var, 
        yerr=y_var_err, 
        label=f'{voltage} V: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        color = color
        )

    mask = mask & (afterpulse_df['datetime'] > pd.Timestamp('2024-10-29'))

    ax[0].errorbar(
        afterpulse_df['datetime'][mask],
        y_var[mask], 
        yerr=y_var_err[mask], 
        fmt='o',
        color = color,
        markerfacecolor = 'white'
        )
    


#rotate axis ticks lable
xlabels = ax[0].get_xticklabels()
ax[0].set_xticklabels(xlabels, rotation=30, ha='right')

for date in np.unique(afterpulse_df['date']):
    mask = afterpulse_df['date'] == date
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = abs(afterpulse_df['voltage'][mask])

    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))

    

    color = next(colors)["color"]

    if (afterpulse_df['run_tag'][mask].iloc[0] == "LXe/tritium"):
        # marker hollow
        markerfacecolor='white'
    else:
        markerfacecolor=color

    ax[1].errorbar(
        x_var,
        y_var, 
        yerr=y_var_err, 
        label=f'{date}: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        color=color,
        mfc=markerfacecolor
    )

ax[0].legend(
    title="bias voltage: correlation",
    bbox_to_anchor=(0.5, 1.),
    loc='upper center',
    frameon=True,
    ncols=2,
    fontsize=14

)


ax[1].legend(
    title="date: correlation",
    bbox_to_anchor=(0.5, 1),
    loc='upper center',
    frameon=True,
    ncols=2,
    fontsize=14

)

plt.ylim(2.5, 8.2)

ax[0].set_ylabel("$\\tau$ [$\mu$s]")
ax[0].set_xlabel("Date")
ax[1].set_xlabel("$|V_{bias}|$ [V]")

fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_dependence.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')

In [ ]:
# can specify the event_id here
# event_id = 6564
event_id = None

if event_id is None:
    event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
    event_id = event_id_array[event_id_id]

print(f"Selected event_id: {event_id}")

single_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=3
extend_sum_window=50
peak_merge_window_sample = 250 # samples
        
window_size = 6
min_peak_width_sample = window_size*3

single_info.set_peaks_for_single_processed_waveform(
                    single_waveform, 
                    single_baseline, 
                    single_baseline_std,
                    threshold_sig=5, 
                    extend_sum_window=50,
                    peak_merge_window_sample=peak_merge_window_sample,
                    show_plot = True,
                    event_id=event_id
                )


fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/example_waveform-{event_id}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')


In [ ]:
peak_boundaries

In [ ]:
mask = selected_data.event_id == event_id
test = selected_data.apply_mask(mask, inplace=False, dry=True)

print(test.peak_area_PE)

In [ ]:
result, edge = np.histogram(d2d_data.peak_height_V, bins = 100, range=[0,0.1])
edge[np.argmax(result)]  # get the bin center of the peak

### Event Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
# this also remove null values from the peak_height_V_array
array = list(d2d_data.peak_start_time_s_array)
peak_start_time = np.concatenate(array)

event_max = np.max(peak_start_time)
event_min = np.min(peak_start_time)
n_bins = int(event_max - event_min) # 1 second per bin

ax.hist(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins, alpha=0.5)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Event Rate [Hz]')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
n_bins = 207


array = list(d2d_data.peak_rel_start_time_s_array)
peak_start_time = np.concatenate(array)
ax.hist(peak_start_time*1e6, alpha=0.5, bins = n_bins,
        label='without merge window'
        )

# this also removes null values from the peak_height_V_array
array = list(d2d_data_2.peak_rel_start_time_s_array)
peak_start_time = np.concatenate(array)
ax.hist(peak_start_time*1e6, alpha=0.5, bins = n_bins,
        label='with 1 us merge window'
        )

ax.set_xlim(0, 4)  # limit x-axis to 1000 us

# log y
ax.set_yscale('log')

ax.set_xlabel('Time [us]')
ax.set_ylabel('Counts')

plt.legend()

plt.show()

In [ ]:
np.unique(d2d_data.board)

### Area vs Time

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
# this also remove null values from the peak_height_V_array
peak_rel_start_time = d2d_data.peak_rel_start_time_s
peak_area = d2d_data.peak_area_Vns
# array = list(d2d_data.peak_rel_start_time_s_array)
# peak_rel_start_time = np.concatenate(array)

# array = list(d2d_data.peak_area_Vns_array)
# peak_area = np.concatenate(array)

# event_max = np.max(peak_start_time)
# event_min = np.min(peak_start_time)
# n_bins = int(event_max - event_min) # 1 second per bin
# ax.hist2d(peak_rel_start_time*1e6, peak_area, bins = n_bins,
#         # range = [event_min,event_min+n_bins], bins = n_bins
#         )

ax.set_xlabel('Time [us]')


log = True
if not log:
        ax.hist2d(peak_rel_start_time*1e6, (peak_area),
                bins=[100,100],
                range=[[0.1,4],[-1,10]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        ax.axhline(1)
        ax.set_ylabel('Area [PE]')

else: 
        ax.hist2d(peak_rel_start_time*1e6, np.log10(peak_area),
                bins=[100,100],
                range=[[0,4],[-1,3]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        ax.axhline(0)
        ax.set_ylabel('log(Area [PE])')



# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/area_time_Cs137.pdf")

In [ ]:
np.unique(d2d_data.channel)

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# this also remove null values from the peak_height_V_array
peak_rel_start_time = d2d_data.peak_rel_start_time_s
peak_area = d2d_data.peak_area_PE
ax.set_xlabel('Time [us]')

log = True
if not log:
        ax.hist2d(peak_rel_start_time*1e6, (peak_area),
                bins=[100,100],
                range=[[0.1,4],[-1,10]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        # ax.axhline(1)
        ax.set_ylabel('Area [PE]')

else: 
        ax.hist2d(peak_rel_start_time*1e6, np.log10(peak_area),
                bins=[100,100],
                range=[[0,4],[-1,3]],
                cmap='viridis',
                norm=LogNorm()
                ) 
        # ax.axhline(0)
        ax.set_ylabel('Peak Area [PE]')

# configure the y axis labels and ticks
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['1', '$10^{1}$', '$10^{2}$', '$10^{3}$'])

# color bar
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Counts')

# grid = True
ax.grid(True, color='black')

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/area_time_nosource.pdf")

#### Afterpulse fit

In [ ]:
df_result['first_peak_start_time'] = df_result.groupby('event_id')['peak_rel_start_time_s'].transform('min')
df_result['time_diff'] = df_result['peak_rel_start_time_s'] - df_result['first_peak_start_time']


fig, ax = plt.subplots(figsize=(12,5))

# peak_rel_start_time = d2d_data.peak_rel_start_time_s
time_diff = df_result['time_diff']

d2d_data.get_df()

ax.set_xlabel('Time [us]')

ax.hist(time_diff*1e6, 
        bins=100,
        range=[0,4],
        alpha = 0.7,
         label='data'
        ) 
ax.set_ylabel('Count')

# log y
ax.set_yscale('log')

hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
hist_log = np.log10(hist + 1)  # add 1 to avoid log(0)
# ax.plot(bin_centers, hist_log, 'b-', label='Data')

# fit the decay trend
mask = (bin_centers >= 1) & (bin_centers <= 2.5)
# linear fit
p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


# find tau
tau = -1 / p[0]
a = np.exp(p[1])
# error of tau
tau_err = np.sqrt(cov[0, 0]) / p[0]**2

# plot the fit line
x = np.linspace(bin_edges[0], bin_edges[-1], 10)
y = p[0] * x + p[1]
# y = np.log(a * np.exp(-x / tau))
ax.plot(x, 10**y, '--', label=f'$\\tau$ = {tau:.2f} $\pm$ {tau_err:.2f} $\mu$s')
ax.legend()

ax.set_ylim(1e3, 2e5)
ax.set_xlim(-0.1, 3)

df_output = pd.DataFrame({
    'voltage': np.unique(df_result.voltage_preamp1_V)[0],
    'tau': tau,
    'error': tau_err
},
index=[0]
)

df_output.to_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse.csv",
    mode="a",
    header=False,
    index=False
)

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_decay_time.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# this also remove null values from the peak_height_V_array
# peak_area = d2d_data.peak_area_Vns

# peak_rel_start_time = d2d_data.peak_rel_start_time_s
time_diff = df_result['time_diff']

d2d_data.get_df()

ax.set_xlabel('Time [us]')

# ax.hist(time_diff*1e6, 
#         bins=100,
#         range=[0,4],
#         ) 
# ax.axhline(1)
ax.set_ylabel('Count')

# log y
# ax.set_yscale('log')

# grid = True
ax.grid(True)

hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
hist_log = np.log(hist + 1)  # add 1 to avoid log(0)
ax.plot(bin_centers, hist_log, 'b-', label='Data')

# fit the decay trend
mask = (bin_centers >= 1) & (bin_centers <= 2.5)
# linear fit
p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


# find tau
tau = -1 / p[0]
a = np.exp(p[1])

# plot the fit line
x = np.linspace(bin_edges[0], bin_edges[-1], 10)
y = np.log(a * np.exp(-x / tau))
ax.plot(x, y, 'r--', label=f'Fit: a*exp(-x/{tau:.2f})')
ax.legend()

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/area_time_nosource.pdf")

In [ ]:
np.exp(1)

In [ ]:
event_rate,bin_edge = np.histogram(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins)

### Estimate Coincidence Window

In [ ]:
DetectorHeight = 100 #mm
DetectorDiameter = 100 #mm
DetectorLongestLength = (DetectorHeight**2 + DetectorDiameter**2)**0.5

SpeedofLight = 299792458 # m/s
CoincidenceTime = DetectorLongestLength / SpeedofLight * 1e9 # in ns

In [ ]:
CoincidenceTime

### Attempt to find the Time difference between board 0 and 1

In [ ]:
mask = d2d_data.board == 0
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

board_0_peak_time = single_board_data.peak_rel_start_time_s

mask = d2d_data.board == 1
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
board_1_peak_time = single_board_data.peak_rel_start_time_s

length = np.min([len(board_0_peak_time), len(board_1_peak_time)])
time_diff = np.abs(board_1_peak_time[:length] - board_0_peak_time[:length]) # in s

# plt.hist(time_diff, bins=50, range = [0, 1e-7])
plt.hist(time_diff, bins=50, range=[1.5e-7,0.5e-6])
plt.show()


In [ ]:
count, edge = np.histogram(time_diff, bins=50, range=[1.5e-7,0.5e-6])
edge[np.argmax(count)]  # get the bin center of the peak

### Area Spectrum

In [ ]:
# @jit(nopython=True)
def get_sum_area_PE_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width_s: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width_s) 
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    sum_area_PE_list = []
    rel_time = []
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width_s

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        area_PE_within_window = peak_area_PE[mask]
        event_id_within_window = event_id[mask]
        event_time_s_within_window = event_time[mask]
        rel_time_within_window = relative_start_time[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            # if (len(np.unique(channels_within_window)) >= coincidence):
            sum_area_PE = np.sum(area_PE_within_window)
            sum_area_PE_list.append(sum_area_PE)
            rel_time.append(np.min(event_time_s_within_window))
            
    return (sum_area_PE_list, rel_time)



In [ ]:
# @jit(nopython=True)
def get_counts_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width_s: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width_s) # 1 second per bin
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width_s

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            counts += 1
            
    return counts



Attempt to find baoard 1 and board 0 delay time

In [ ]:
# change peak_start_time_s for all board 1 in d2d_data
count_list = []
delay_time_list = np.arange(-10,10,1)

for delay_time in delay_time_list:
    change_time = d2d_data.get_df()
    change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
    d2d_data_updated = d2d.data(change_time)
    mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5)
    clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

    # mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
    # clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    counts = get_counts_in_time_window(
        clean_data.peak_start_time_s, 
        clean_data.peak_rel_start_time_s,
        clean_data.event_start_time_s, 
        clean_data.channel, 
        clean_data.board, 
        clean_data.peak_area_PE, 
        clean_data.event_id,
        time_window_width_s = 1,  # 1 us
        coincidence=2)
    count_list.append(counts)


In [ ]:
delay_time_list[np.argmax(count_list)]  # get the delay time with the maximum count

In [ ]:
plt.scatter(delay_time_list, count_list)

#### Summed spectrum

In [ ]:
# delay_time = -1.9
delay_time = 0

change_time = d2d_data.get_df()
change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
d2d_data_updated = d2d.data(change_time)

mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5) 
# & (d2d_data_updated.peak_height_V < 1.2)
clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

# mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
# clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

sum_area_PE_list, rel_time = get_sum_area_PE_in_time_window(
    clean_data.peak_start_time_s, 
    clean_data.peak_rel_start_time_s,
    clean_data.event_start_time_s, 
    clean_data.channel, 
    clean_data.board, 
    clean_data.peak_area_PE, 
    clean_data.event_id,
    time_window_width_s = 0.0000001,  # 1 us
    coincidence=3)

In [ ]:

mask = (d2d_data_2.peak_area_PE > d2d_data_2.spe_position*1.5) 
# & (d2d_data_updated.peak_height_V < 1.2)
clean_data = d2d_data_2.apply_mask(mask, inplace=False, dry=True)

# mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
# clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    clean_data.peak_start_time_s, 
    clean_data.peak_rel_start_time_s,
    clean_data.event_start_time_s, 
    clean_data.channel, 
    clean_data.board, 
    clean_data.peak_area_PE, 
    clean_data.event_id,
    time_window_width_s = 0.0000001,  # 1 us
    coincidence=3)

In [ ]:
plt.hist(sum_area_PE_list_0, bins=100
         , range=[-0.1,3000]
)

plt.hist(sum_area_PE_list, bins=100
         , range=[-0.1,3000]
)


plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
plt.yscale('log')

#### Proxy(Event Rate) for both boards

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
        #  , range = [49,51]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
        #  , range = [49,51]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
         , range = [32,34]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
         , range = [32,34]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
plt.hist(rel_time_0, bins=100
)
         
#log y
plt.yscale('log')

In [ ]:
# double check...

plt.close()
fig, ax = plt.subplots(figsize=(10,6))

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    ax.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Time coincidence

In [ ]:
# double check...



plt.close()
fig, ax = plt.subplots(figsize=(10,6))

result_list = []

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    np.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Test

In [ ]:
event_id = 3000
single_waveform = waveform[event_id,:]

baseline, baseline_std = get_baseline_for_all_events(waveform)
single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

event_time_s = waveform_processor.event_time_s[event_id]


In [ ]:
plt.scatter(points,single_waveform[points], color='r', label='boundaries of peaks', zorder=10)
plt.plot(single_waveform, label='filtered waveform')
plt.plot(smooth_waveform, label='waveform with rolling window')
plt.hlines(threshold, 0, 1000, 'black', linestyles='--', label='threshold')

plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')

In [ ]:
pairs = np.array([[15, 27], [38, 50], [60, 80]])
peak_width = pairs[:,1] - pairs[:,0]
tmp = np.where(peak_width < min_peak_width_sample)

pairs = np.delete(pairs, tmp, axis=0)
pairs

In [ ]:
min_peak_width_sample

In [ ]:
v_get_waveform = np.vectorize(
    get_peaks, 
    excluded=['window_size', 'threshold_sig', 'peak_width_sample'], 
    signature="(n) -> ()")

In [ ]:
single_waveform.shape

#### peak finding (nd version)

In [ ]:
axis = 1
window_size = 6
threshold_sig = 3
event_id = 3000

# single_waveform = waveform[event_id,:]
# single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

smooth_waveform = rolling_window(waveform, window_size, axis=1)
    
# add rolling window to smooth the data, roll every 3 points
# test_averaged = np.convolve(waveform, np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
# baseline, baseline_std = get_baseline_for_all_events(smooth_waveform)
threshold = baseline + threshold_sig * baseline_std

threshold = np.repeat(threshold, smooth_waveform.shape[1]).reshape(smooth_waveform.shape[0], -1)
mask = smooth_waveform > threshold

# find where the waveform passes the threshold
diff = np.diff(mask, axis = 1)

points = np.where(diff == 1)
event_id, array_idx, count = np.unique(points[0], return_counts=True, return_index=True)

# remove the last point if it's odd
odd_points = np.where(count % 2 != 0)[0]

# since all repeating points are consecutive, so we can just add the count to the index
# This will give us the end index of the last point
last_index = array_idx + count - 1
last_index_to_remove = last_index[odd_points]

# remove the last point if it's odd
peaks_position_event = np.delete(points[0], last_index_to_remove)
peaks_position_sample = np.delete(points[1], last_index_to_remove)

# double check if the peaks_position_event is odd
event_id, array_idx, count = np.unique(peaks_position_event, return_counts=True, return_index=True)
assert len(np.where(count % 2 != 0)[0]) == 0, "There are still odd points in the peaks_position_event array."


In [ ]:
test_event_id = 5000
start_idx = array_idx[test_event_id]
end_idx = array_idx[test_event_id] + count[test_event_id]
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(threshold[test_event_id], 'b--', label='threshold')
ax.plot(mask[test_event_id], 'g--', label='threshold')
ax.plot(waveform[test_event_id,:], label='smoothed waveform')
ax.plot(smooth_waveform[test_event_id,:], label='smoothed waveform')

ax.plot(peaks_position_sample[start_idx:end_idx],
        test[start_idx:end_idx], 'ro', label='start of peak')

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)
event_id

#### peak finding (1d version)

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)[0]

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=5
extend_sum_window=50
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
print(threshold)


# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)



fig, ax = plt.subplots(figsize=(10,6))
print(f"Threshold: {threshold:.3f} V")
# ax.plot(mask, 'g--', label='threshold')
ax.plot(single_processed_waveform, label='raw waveform')
ax.plot(smooth_waveform, label='smoothed waveform')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')


for peak_id, peak_boundary in enumerate(peak_boundaries):
    while (extend_sum_window > 0):
        start_sample = np.max([peak_boundary[0]-extend_sum_window, 0])
        end_sample = np.min([peak_boundary[1]+extend_sum_window, len(single_processed_waveform)])

        # remove the pair if they are too far from the baseline
        y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
        if y_diff > threshold_sig*single_baseline_std:
            extend_sum_window -= 10
        else:
            break
    else:
        # if we cannot find a valid window, just use the original peak boundary
        start_sample = peak_boundary[0]
        end_sample = peak_boundary[1]

    ax.plot([start_sample,end_sample],
        [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
        'o', label = f"Peak {peak_id}")

plt.title(f"Event {event_id}")
plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')
plt.show()

In [ ]:
y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
y_diff

### Test

In [ ]:
axis = 1
window_size = 4
test = np.ones((99,200))
# rolled_array = np.lib.stride_tricks.sliding_window_view(test, 4, axis=1).mean(axis=1)
rolled_array = np.lib.stride_tricks.sliding_window_view(test, window_size, axis=axis)
test_mean = rolled_array.mean(axis=test.ndim) # this is the same as the rolling mean
# print(rolled_array)
print(test_mean.shape)
# rolled_array

### Old waveform check 

In [ ]:
# can specify the event_id here
# event_id = 16641
# event_id = 8623
event_id = 1016
# event_id = None

if event_id is None:
    event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
    event_id = event_id_array[event_id_id]

print(event_id)

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=3
extend_sum_window=50
peak_merge_window_sample = 250 # samples
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
# threshold = 0.015

# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)

ax.plot(single_processed_waveform, label='waveform')
ax.axhline(single_baseline, color='green', linestyle = 'dashdot', label='baseline')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')





# ax.plot(smooth_waveform, label='2. smoothed waveform')
# ax.scatter(peak_boundaries, single_processed_waveform[peak_boundaries],  s = 50, marker = '*', color = 'black', label='3. detected peaks above threshold', zorder = 10)
# print(f"Found {len(peak_boundaries)} peaks in event {event_id}.")


# avoid overlapping peaks
end_sample_of_previous_peak = 0

for peak_id, peak_boundary in enumerate(peak_boundaries):

    # find the first sample below the baseline before the peak boundary
    start_sample = peak_boundary[0] - np.where(single_processed_waveform[peak_boundary[0]::-1] - single_baseline < 0)[0][0]
    # find the first sample above the baseline after the peak boundary
    end_sample = peak_boundary[1] + np.where(single_processed_waveform[peak_boundary[1]:] - single_baseline < 0)[0][0]
    
    # skip if the start sample is before the end sample of the previous peak
    if start_sample < end_sample_of_previous_peak:
        continue

    end_sample_of_previous_peak = end_sample    
    
    peak_area_Vsample = np.sum(single_processed_waveform[start_sample:end_sample])
    peak_area_Vns = peak_area_Vsample * 4  # convert to V*ns, assuming 250 MHz sampling rate (4 ns per sample)  
    peak_area_PE = peak_area_Vns / single_info.spe_position
    peak_width_ns = (end_sample - start_sample)*4
    peak_height_V = np.max(single_processed_waveform[start_sample:end_sample])

    print(f"Peak {peak_id}:"
          f"peak_height_V = {peak_height_V:.3f}, peak_area_PE = {peak_area_PE:.3f}, "
          f"peak_width_ns = {peak_width_ns:.3f} \n")

    # ax.plot([start_sample,end_sample],
    #     [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
    #     'o', label = f"Peak {peak_id}, area[PE] = {peak_area_PE:.1f}")
    
    # plot the found peaks in shaded area
    ax.fill_between(np.arange(start_sample, end_sample),
                    single_processed_waveform[start_sample:end_sample],
                      alpha=0.5, label=f"area = {peak_area_PE:.1f} PE")



# # merge peaks within the peak_merge_window
# n_peaks = len(peak_boundaries)
# for iterations in np.arange(0, n_peaks-1):
#     if len(peak_boundaries) < 2:
#         break

#     if len(peak_boundaries) <= iterations + 1:
#         ax.axvspan(peak_boundaries[iterations,0], peak_boundaries[iterations,1],
#                 # np.min(single_processed_waveform),
#                 # np.max(single_processed_waveform),
#                 color='grey',
#                 alpha=0.5, label=f"merge peak", zorder=1)
#         break

#     peak_window_boundary = peak_boundaries[iterations,0] + peak_merge_window_sample
#     idxs = np.where(peak_boundaries[1:,0] < peak_window_boundary)[0] # idxs is shifted by 1 due to the slicing
#     if len(idxs) == 0:
#         end_boundary = peak_boundaries[iterations,1]
#     else:
#         end_boundary = peak_boundaries[idxs[-1]+1,1]
#         peak_boundaries = np.delete(peak_boundaries, idxs+1, axis=0)
#         peak_boundaries[iterations,1] = end_boundary
#     ax.axvspan(peak_boundaries[iterations,0], end_boundary,
#                 # np.min(single_processed_waveform),
#                 # np.max(single_processed_waveform),
#                 color='grey',
#                 alpha=0.5, label=f"merge peak", zorder=1)


# merge peaks within the peak_merge_window
n_peaks = len(peak_boundaries)

for iterations in np.arange(0, n_peaks-1):
    if len(peak_boundaries) < 2:
        break

    if len(peak_boundaries) <= iterations + 1:
        break

    peak_window_boundary = peak_boundaries[iterations,0] + peak_merge_window_sample
    idxs = np.where(peak_boundaries[iterations + 1:,0] < peak_window_boundary)[0] # idxs is shifted by 1 due to the slicing
    if len(idxs) > 0:
        end_boundary = peak_boundaries[iterations + 1 + idxs[-1],1]
        peak_boundaries = np.delete(peak_boundaries, idxs+1, axis=0)
        peak_boundaries[iterations,1] = end_boundary
    ax.axvspan(peak_boundaries[iterations,0], end_boundary,
                # np.min(single_processed_waveform),
                # np.max(single_processed_waveform),
                color='grey',
                alpha=0.5, label=f"merge peak", zorder=1)

plt.title(f"Event {event_id}")
plt.legend(fontsize='15', bbox_to_anchor=(0, 1), loc='upper left')
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')

fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/example_waveform-{event_id}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
